## ARIMA-Modellvergleich 

**Zeitreihen:**
Luftdruck (A): ARIMA(p_luft, 1, q_luft)  – aus Grid-Search auf Luftdruck
Wind (B): ARIMA(1, 1, 2)            – Grid-Search auf Windgeschwindigkeit
Temperatur (C): ARIMA(1, 1, 2)            – BIC-Kriterium auf Temperatur

**Evaluationsstrategie:**
1. Train/Test-Split 70/30 (Multi-Step-Prognose)
2. 5-Fold Time-Series-Cross-Validation (30 Tage pro Fold)
3. Metriken: RMSE, MAE, MAPE

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import itertools

# DATEIPFADE
PFAD_LUFTDRUCK  = "/Users/clara/Desktop/uni_dreck/Menden_S26/luftdruck_bereinigt.csv"
PFAD_WIND       = "/Users/clara/Desktop/uni_dreck/Menden_S26/wind_bereinigt.csv"
PFAD_TEMPERATUR = "/Users/clara/Desktop/uni_dreck/Menden_S26/temperatur_bereinigt.csv"

# DATEN LADEN
def lade_csv(pfad, wertspalte):
    df = pd.read_csv(pfad, parse_dates=["datum"], index_col="datum")
    ts = df[wertspalte].asfreq("D")
    return ts.interpolate(method="time")

print("Lade Zeitreihen ...")
ts_luft = lade_csv(PFAD_LUFTDRUCK,  "luftdruck_hpa")
ts_wind = lade_csv(PFAD_WIND,       "windgeschwindigkeit_ms")
ts_temp = lade_csv(PFAD_TEMPERATUR, "temperatur_mittel_c")

print(f"Luftdruck : {ts_luft.index[0].date()} bis {ts_luft.index[-1].date()}  ({len(ts_luft):,} Tage)")
print(f"Wind      : {ts_wind.index[0].date()} bis {ts_wind.index[-1].date()}  ({len(ts_wind):,} Tage)")
print(f"Temperatur: {ts_temp.index[0].date()} bis {ts_temp.index[-1].date()}  ({len(ts_temp):,} Tage)")

# MODELLE DEFINIEREN
print("\nGrid Search: Bestes Luftdruck-Modell ...")
ergebnisse = []
for p, q in itertools.product(range(4), range(4)):
    if p == 0 and q == 0:
        continue
    try:
        fit = ARIMA(ts_luft, order=(p, 1, q)).fit()
        ergebnisse.append({"p": p, "q": q, "AIC": fit.aic})
    except Exception:
        pass
erg_df = pd.DataFrame(ergebnisse).sort_values("AIC")
p_A, q_A = int(erg_df.iloc[0]["p"]), int(erg_df.iloc[0]["q"])
print(f"Bestes Luftdruck-Modell: ARIMA({p_A},1,{q_A})")

MODELLE = {
    f"Modell_A  ARIMA({p_A},1,{q_A})": (p_A, 1, q_A),   # Luftdruck (Grid-Search)
    "Modell_B  ARIMA(1,1,2)":           (1,   1, 2),       # Wind
    "Modell_C  ARIMA(1,1,2)":           (1,   1, 2),       # Temperatur
}

ZEITREIHEN = {
    "Luftdruck":  ts_luft,
    "Wind":       ts_wind,
    "Temperatur": ts_temp,
}

# EVALUATIONSFUNKTIONEN
def metriken(ist, prognose):
    rmse = np.sqrt(mean_squared_error(ist, prognose))
    mae  = mean_absolute_error(ist, prognose)
    with np.errstate(divide="ignore", invalid="ignore"):
        mape = np.mean(np.abs((np.array(ist) - np.array(prognose)) / np.array(ist))) * 100
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

def eval_train_test(ts, order):
    n = int(len(ts) * 0.70)
    train, test = ts.iloc[:n], ts.iloc[n:]
    try:
        fc = ARIMA(train, order=order).fit().get_forecast(steps=len(test)).predicted_mean
        fc.index = test.index
        return metriken(test.values, fc.values)
    except Exception:
        return {"RMSE": np.nan, "MAE": np.nan, "MAPE": np.nan}

def eval_cv(ts, order, n_splits=5, test_size=30, fenster=730):
    ts_cv = ts.iloc[-min(fenster, len(ts)):]
    tscv  = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    fold_metriken = []
    for _, (idx_tr, idx_te) in enumerate(tscv.split(ts_cv)):
        try:
            fc = ARIMA(ts_cv.iloc[idx_tr], order=order).fit() \
                     .get_forecast(steps=len(idx_te)).predicted_mean
            fc.index = ts_cv.iloc[idx_te].index
            fold_metriken.append(metriken(ts_cv.iloc[idx_te].values, fc.values))
        except Exception:
            pass
    if not fold_metriken:
        return {"RMSE": np.nan, "MAE": np.nan, "MAPE": np.nan}
    df = pd.DataFrame(fold_metriken)
    return {k: df[k].mean() for k in ["RMSE", "MAE", "MAPE"]}

# EVALUATION
print("\n" + "="*65)
print("EVALUATION: 3 Modelle x 3 Zeitreihen")
print("="*65)

records_tt, records_cv = [], []

for modell_name, order in MODELLE.items():
    for ts_name, ts in ZEITREIHEN.items():
        print(f"  {modell_name}  x  {ts_name} ...")
        m_tt = eval_train_test(ts, order)
        m_cv = eval_cv(ts, order)
        records_tt.append({"Modell": modell_name, "Zeitreihe": ts_name, **m_tt})
        records_cv.append({"Modell": modell_name, "Zeitreihe": ts_name, **m_cv})

df_tt = pd.DataFrame(records_tt)
df_cv = pd.DataFrame(records_cv)

# ERGEBNISTABELLEN
print("\n" + "="*65)
print("RMSE - Train/Test (70/30)")
print("="*65)
print(df_tt.pivot(index="Modell", columns="Zeitreihe", values="RMSE").round(4).to_string())

print("\n" + "="*65)
print("MAE - Train/Test (70/30)")
print("="*65)
print(df_tt.pivot(index="Modell", columns="Zeitreihe", values="MAE").round(4).to_string())

print("\n" + "="*65)
print("RMSE - 5-Fold CV")
print("="*65)
print(df_cv.pivot(index="Modell", columns="Zeitreihe", values="RMSE").round(4).to_string())

print("\n" + "="*65)
print("MAE - 5-Fold CV")
print("="*65)
print(df_cv.pivot(index="Modell", columns="Zeitreihe", values="MAE").round(4).to_string())

print("\n" + "="*65)
print("MAPE % - 5-Fold CV")
print("="*65)
print(df_cv.pivot(index="Modell", columns="Zeitreihe", values="MAPE").round(4).to_string())

# RANG & EMPFEHLUNG
def rang(pivot):
    return pivot.rank(axis=0, ascending=True).astype(int)

pivot_cv = df_cv.pivot(index="Modell", columns="Zeitreihe", values="RMSE")
rang_cv  = rang(pivot_cv).mean(axis=1).sort_values()

print("\n" + "="*65)
print("MITTLERER RMSE-RANG (CV) - 1 = bestes Modell")
print("="*65)
for modell, r in rang_cv.items():
    marker = " <- EMPFOHLEN" if modell == rang_cv.index[0] else ""
    print(f"  {modell}: {r:.2f}{marker}")

Lade Zeitreihen ...
Luftdruck : 1966-01-01 bis 2026-01-11  (21,926 Tage)
Wind      : 1966-01-01 bis 2026-01-11  (21,926 Tage)
Temperatur: 1966-01-01 bis 2026-01-11  (21,926 Tage)

Grid Search: Bestes Luftdruck-Modell ...


/Users/clara/Library/Python/3.13/lib/python/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Bestes Luftdruck-Modell: ARIMA(3,1,2)

EVALUATION: 3 Modelle x 3 Zeitreihen
  Modell_A  ARIMA(3,1,2)  x  Luftdruck ...
  Modell_A  ARIMA(3,1,2)  x  Wind ...
  Modell_A  ARIMA(3,1,2)  x  Temperatur ...


/Users/clara/Library/Python/3.13/lib/python/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Modell_B  ARIMA(1,1,2)  x  Luftdruck ...
  Modell_B  ARIMA(1,1,2)  x  Wind ...
  Modell_B  ARIMA(1,1,2)  x  Temperatur ...
  Modell_C  ARIMA(1,1,2)  x  Luftdruck ...
  Modell_C  ARIMA(1,1,2)  x  Wind ...
  Modell_C  ARIMA(1,1,2)  x  Temperatur ...

RMSE - Train/Test (70/30)
Zeitreihe               Luftdruck  Temperatur    Wind
Modell                                               
Modell_A  ARIMA(3,1,2)     8.2138     11.7407  1.6241
Modell_B  ARIMA(1,1,2)     8.2139     11.7457  1.6240
Modell_C  ARIMA(1,1,2)     8.2139     11.7457  1.6240

MAE - Train/Test (70/30)
Zeitreihe               Luftdruck  Temperatur    Wind
Modell                                               
Modell_A  ARIMA(3,1,2)     6.3038      9.7312  1.2553
Modell_B  ARIMA(1,1,2)     6.3045      9.7361  1.2551
Modell_C  ARIMA(1,1,2)     6.3045      9.7361  1.2551

RMSE - 5-Fold CV
Zeitreihe               Luftdruck  Temperatur    Wind
Modell                                               
Modell_A  ARIMA(3,1,2)     7.39

Hier kann man sehen, dass die Modelle gleich gut funktionieren. Das Model ARIMA(1,1,2) ist sowohl bei Temperatur als auch Wind das beste. Deswegen würde ich dieses nutzen. 